# **ALGORITHMIC CONSTRUCTION OF BAKER-AKHIEZER FUNCTIONS**

In [15]:
import sys
sys.path.insert(0, "..") # dalgebra is here

from dalgebra import *
from dalgebra.commutators import *

%display latex

## 1. Paper example (Examples 1.1 and 2.2)

In this example we consider an order 3 differential operator 
$$L = \partial^3 - 6\wp \partial -6\wp',$$
where $\wp = \wp(x;0,1)$ is the $\wp$-Weierstrass function that satisfies the differential equation $\wp'^2 = 4\wp^3 - 1$.

In [16]:
g_2, g_3 = 0,1
R.<eta> = DElliptic(DifferentialRing(QQ), f"eta_p^2 - 4*eta^3 - {g_2}*eta - {g_3}")
eta_p = eta.derivative()
DO.<z> = DifferentialPolynomialRing(R)
L = z[3] - 6*eta*z[1] - 6*eta_p*z[0]
L

-(6*eta_p)*z_0 - (6*eta)*z_1 + z_3

### Example 1.1: the centralizer

We now compute its centralizer. We knew this centralizer was not trivial and that the level of $L$ is exactly 4. Hence the following line computes the complete centralizer:

In [17]:
_,G,_ = GetCentralizer([L.coefficient_full(z[i]) for i in range(L.order(z)-1)], 4, starting_level=4, ignore_bound=True)

In [18]:
G[1]

-(24*eta^2)*z_0 - (12*eta_p)*z_1 - (8*eta)*z_2 + z_4

In [19]:
G[2]

-(40*eta*eta_p)*z_0 - (80*eta^2)*z_1 - (20*eta_p)*z_2 - (10*eta)*z_3 + z_5

### Example 2.2: the spectral curve

From this basis as a $C[L]$-module, we can compute the Burchnall-Chaundy ideal for $L$ (i.e., the ideal that define all algebraic relations between $L$ and the generators of the centralizer). This can be achieved with the method `BC_ideal`:

In [20]:
I = BC_ideal(L, G[1:], gen=z)
for el in I.gens():
    show(el)

mu_1^2 - mu_2*lambda_

mu_1*mu_2 - lambda_^3 + 4*lambda_

mu_2^2 - mu_1*lambda_^2 + 4*mu_1

From this ideal, we can study the spectral curve of $L$. For example, we can check its genus, showing that in this case the spectral curve is not rational nor elliptic:

In [21]:
I.genus()

2

### Example 2.2: the common right factor

In order to compute the Baker-Akhiezer function, i.e., the common eigenfunction for all operators in the centralizer of $L$, we need to compute the common right factor of the operators in the centralizer. We can achieve this with the following code:

In [22]:
sp_ops = spectral_operators(*G[1:], L, names=["mu_1","mu_2","lambda_"])
G_1, G_2, L_s = sp_ops
DO_S = L_s.parent()
z_s = DO_S.gen("z")

In [23]:
%%time
subres_seq = DO_S.sylvester_subresultant_sequence(L_s, G_1, z_s)

CPU times: user 35 ms, sys: 0 ns, total: 35 ms
Wall time: 33.5 ms


The first subresultant is a constant polynomial that is in the BC ideal:

In [24]:
subres_seq[0]

(lambda_^4 - mu_1^3 - 4*lambda_^2)*z_0

In [25]:
subres_seq[0].reduce_algebraic(I)

0

On the other hand, the first subresultant is a non-zero first order differential operator that coincides with the greatest common right divisor of $L$ and $G_1$:

In [26]:
subres_seq[1]

(2*mu_1*eta_p + 8*lambda_*eta^2 - mu_1*lambda_)*z_0 - (2*lambda_*eta_p + 2*mu_1*eta - lambda_^2)*z_1

### Example 2.2: computing the Baker-Akhiezer function

To compute the Baker-Akhiezer function, we simply need to solve the equation induced by the gcrd of $L$ and $G_1$. It is an equation of the form
$$c_1 \Psi' + c_0 \Psi = 0,$$
so we know that a solution can be obtained as
$$\Psi = \exp\left(\int\frac{-c_0}{c_1}\right) = \exp(\Phi).$$

In [27]:
c_0, c_1 = subres_seq[1].coefficients()

In [28]:
-c_0/c_1

((eta^2 + ((-mu_1^2)/(4*lambda_^2))*eta)/(eta^3 + (4*mu_1^2/(-16*lambda_^2))*eta^2 + 1/4*mu_1*eta - 1/16*lambda_^2 + 1/4))*eta_p + (1/2*lambda_*eta^2 + 1/2*mu_1^2/(4*lambda_)*eta + (1/8*mu_1*lambda_^2 - 1/2*mu_1)/(-2*lambda_))/(eta^3 + (4*mu_1^2/(-16*lambda_^2))*eta^2 + 1/4*mu_1*eta - 1/16*lambda_^2 + 1/4)

Final computations require to integrate in the field with the $\wp$-Weierstrass function. We used Maple to solve the previous integral. To simplify the notation, let $d(\wp)$ the denominator of the previous coefficients and let $\alpha_1,\alpha_2,\alpha_3$ be its three roots. Then we can write:

$$\Phi = 
-\frac{1}{4}\left(\sum_{i=1}^3 \frac{
        \left(4\alpha_i^2\lambda^2 - \alpha_i\mu_2\lambda\right)\log(\wp(x) - \alpha_i)
    }{
        16\alpha_i^3 \lambda^2 - 3\alpha_i^2\mu_2\lambda - 2 \alpha_i\lambda^2\mu_1
    }\right)
+ \frac{\lambda}{16}\displaystyle\sum_{i=1}^3 \frac{1}{16\alpha_i^3 \lambda^2 - 3\alpha_i^2\mu_2\lambda - 2 \alpha_i\lambda^2\mu_1} \left(
    \left(8\alpha_i^2\lambda^2 + 2\alpha_i\mu_2\lambda - \lambda^2\mu_1-4\mu_1\right) 
    \left(\log\left(\frac{\sigma(x-\wp^{-1}(\alpha_i); 0,1)}{\sigma(x+\wp^{-1}(\alpha_i); 0,1)}\right) + 2x\zeta(\wp^{-1}(\alpha_i); 0,1)\right)
\right)$$

## 2. (EXTRA) Zeglov's example

In their [paper](https://link.springer.com/article/10.1134/S1995080217060117), the authors proposed a couple of commuting operators of order 4 and 6 with rational functions. In their paper, the rank of this pair is proved to be two (clearly, since the $\gcd$ of the orders is 2). These operators started with the order $4$ operator:
$$L = \partial^4 - 20\frac{x^3(x^5 - 120)}{(x^5 + 30)^2}\partial^2 - 3000\frac{x^2(7x^5 - 90)}{(x^5 + 30)^3}\partial + 18000\frac{x(3x^{10} - 145x^5+450)}{(x^5+30)^4}.$$
However, we have shown in [our paper](https://arxiv.org/pdf/2505.01289) that the operator of order $4$, we showed that $L$ was, in reality, an algebro-geometric operator. It is true that the order 6 operator provided in the first paper commutes with $L$. However, there are other operators commuting with it.

Let us provide Goodearl's basis for this operator:

In [29]:
C = DifferentialRing(QQ)
B = DifferentialRing(QQ[x], [1]); B.set_constant(C); F = B.fraction_field()
x = F('x')
DO.<z> = DifferentialPolynomialRing(F)

In [30]:
y = (x^5 + 30)
L = z[4] - 20*(x**3*(x**5-120))/y^2*z[2] - 3000*(x**2*(7*x**5-90))/y^3*z[1] + 18000*(x*(3*x**10-145*x**5+450))/y^4 * z[0]

In [31]:
%%time
_,G,_ = GetCentralizer(tuple(L.coefficient_full(z[i]) for i in range(L.order(z)-1)), 10,starting_level=1,ignore_bound=True)

CPU times: user 8.89 s, sys: 8.33 ms, total: 8.89 s
Wall time: 8.9 s


In [32]:
[el.order(z) for el in G]

[0, 6, 7, 9]

Hence, it is true that the level of $L$ is 6, but its rank is not 2, but 1. This means this operator is algebro-geometric. Let us compute now the BC-ideal for this operator:

In [33]:
%time
I = BC_ideal(L, G[1:], z)
for el in I.gens():
    show(el)

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 4.05 µs


mu_1^2 - lambda_^3

mu_1*mu_2 - mu_3*lambda_

mu_1*mu_3 - mu_2*lambda_^2

mu_2^2 - mu_1*lambda_^2

mu_2*mu_3 - lambda_^4

mu_3^2 - mu_1*lambda_^3

It is a pretty simple spectral curve. As we know, the generators of the BC-ideal are given by bi-products of two $\mu_i$ variables ($\mu_1$ associated with the operator of order $6$, $\mu_2$ with the operator of order $7$ and $\mu_3$ with the operator of order $9$). We can even check these identities:

In [34]:
G[1].dot(G[2], z) - G[3].dot(L,z) # mu_1*mu_2 - mu_3*lambda

0

We can use the differential subresultant and the curve to compute the greatest common right divisor over the curve of the spectral operators of order $4$ and $7$. This factor will be common to all the operators and, hence, will provide the same solution.

In [35]:
sp_ops = spectral_operators(*G[1:], L, names=["mu_1","mu_2","mu_3","lambda_"])
G_1, G_2, G_3, L_s = sp_ops
DO_S = L_s.parent()
z_s = DO_S.gen("z")

In [36]:
G_2.order(z_s)

7

In [37]:
%%time
subres_seq = DO_S.sylvester_subresultant_sequence(L_s, G_2, z_s)

CPU times: user 40.2 s, sys: 26.9 ms, total: 40.2 s
Wall time: 40.2 s


In [38]:
subres_seq[0].reduce_algebraic(I)

0

In [39]:
RF = subres_seq[1].reduce_algebraic(I)
RF

((-x^15*mu_2*lambda_^3 + 5*x^12*lambda_^4 + 20*x^11*mu_2*lambda_^2 - 90*x^10*mu_2*lambda_^3 + 300*x^8*lambda_^3 + 2550*x^7*lambda_^4 + 7200*x^6*mu_2*lambda_^2 - 2700*x^5*mu_2*lambda_^3 + 11250*x^5*mu_1*lambda_^2 - 13500*x^3*lambda_^3 - 40500*x^2*lambda_^4 - 27000*x*mu_2*lambda_^2 - 27000*mu_2*lambda_^3)/(x^15 + 90*x^10 + 2700*x^5 + 27000))*z_0 + ((x^10*mu_1*lambda_^3 - 5*x^8*lambda_^4 - 20*x^7*mu_2*lambda_^2 - 45*x^6*mu_1*lambda_^2 + 60*x^5*mu_1*lambda_^3 + 150*x^4*lambda_^3 + 600*x^3*lambda_^4 + 900*x^2*mu_2*lambda_^2 + 900*x*mu_1*lambda_^2 + 900*mu_1*lambda_^3)/(x^10 + 60*x^5 + 900))*z_1

#### Checking the right factor from the original pair

We can see that this common-right factor is also a right factor on the common factor of order 2 from the original pair:

In [40]:
%%time
subres_seq_46 = DO_S.sylvester_subresultant_sequence(L_s, G_1, z_s)

CPU times: user 22 s, sys: 0 ns, total: 22 s
Wall time: 22 s


The first two subresultants vanishes on the curve (as they should). The third subresultant is non-zero on the curve and it is an order 2 differential operator:

In [41]:
RF_46 = subres_seq_46[2].reduce_algebraic(I)
RF_46

((-x^20*mu_1*lambda_ - 10*x^18*lambda_^2 + 20*x^16*mu_1 - 120*x^15*mu_1*lambda_ + 200*x^14*lambda_ + 600*x^13*lambda_^2 + 7800*x^11*mu_1 - 5400*x^10*mu_1*lambda_ - 78000*x^9*lambda_ + 63000*x^8*lambda_^2 + 189000*x^6*mu_1 - 108000*x^5*mu_1*lambda_ - 270000*x^4*lambda_ + 1080000*x^3*lambda_^2 - 810000*x*mu_1 - 810000*mu_1*lambda_)/(x^20 + 120*x^15 + 5400*x^10 + 108000*x^5 + 810000))*z_0 + ((-20*x^12*mu_1 - 200*x^10*lambda_ + 300*x^7*mu_1 + 9000*x^5*lambda_ + 27000*x^2*mu_1)/(x^15 + 90*x^10 + 2700*x^5 + 27000))*z_1 + ((x^10*lambda_^2 - 100*x^6*lambda_ + 60*x^5*lambda_^2 + 900*lambda_^2)/(x^10 + 60*x^5 + 900))*z_2

We can compute the differential resultant of this order 2 operator and our common right factor of order 1 and see that it vanishes on the curve. Hence they share a common factor on the right. Since the original right factor had order 1, it must be it:

In [42]:
RF.sylvester_resultant(RF_46, z_s).reduce_algebraic(I) ## It has a common factor --> it has to be RF

0

#### Parametrizing the curve

The curve defined by the BC-ideal happens to be of genus 0 (i.e., it is a rational curve). We hope to parametrize it with rational functions. In fact, due to its shape of having two monomials, we hope to find a parametrization where 
$$\lambda \mapsto t^d,\quad \mu_1 \mapsto t^{d'},\quad \mu_2 \mapsto t^{d''},\quad \mu_3 \mapsto t^{d'''}.$$

In [43]:
I.genus()

0

This leads to a set of linear equations for the values of $d$, $d'$, $d''$ and $d'''$:

In [44]:
for el in I.gens():
    show(el)

mu_1^2 - lambda_^3

mu_1*mu_2 - mu_3*lambda_

mu_1*mu_3 - mu_2*lambda_^2

mu_2^2 - mu_1*lambda_^2

mu_2*mu_3 - lambda_^4

mu_3^2 - mu_1*lambda_^3

To build the system, we just traslate each monomial to a linear term, where we add the degrees of each variable multiplied by its degree. Hence:
* $\mu_1^2 - \lambda^3 \longrightarrow 2d' = 3d$
* $\mu_1\mu_2 - \mu_3\lambda \longrightarrow d'+d'' = d+ d'''$
* $\mu_1\mu_3 - \mu_2\lambda^2 \longrightarrow d'+d''' = d''+2d$
* $\mu_2^2 - \mu_1\lambda^2 \longrightarrow 2d'' = d'+2d$
* $\mu_2\mu_3 - \lambda^4 \longrightarrow d''+d''' = 4d$
* $\mu_3^2 - \mu_1\lambda^3 \longrightarrow 2d''' = d'+3d$

We can build this system and solve it with SageMath:

In [45]:
SYS = Matrix([
    [3, -2, 0, 0], 
    [1, -1, -1, 1], 
    [2, -1, 1, -1],
    [2, 1, -2, 0],
    [4, 0, -1, -1],
    [3, 1, 0, -2]
])

In [46]:
SYS.right_kernel()

Free module of degree 4 and rank 1 over Integer Ring
Echelon basis matrix:
[4 6 7 9]

There is a dimension of possibilities, where we have that $\lambda \mapsto t^4$, $\mu_1 \mapsto t^6$, $\mu_2 \mapsto t^7$ and $\mu_3 \mapsto t^9$. This is not a surprise, since the BC-ideal has at least two terms of precise weight of the given order, hence the order for each variable is reasonable to have for the parametrization.

We proceed to build the ring where these rtional functions live over the parameter $t$:

In [47]:
R_P = DifferentialRing(QQ['x','t'], [1,0])
F_P = R_P.fraction_field()
x_p, t_p = F_P.gens()
DO_P = DifferentialPolynomialRing(F_P, "z")
z_p = DO_P.gen("z")
mor = RF.parent().base().hom([x_p, t_p^6, t_p^7, t_p^9, t_p^4], F_P) 

In [48]:
from dalgebra.dpolynomial.dpolynomial import DPolynomial_Base2BaseMorphism
par_equ = DPolynomial_Base2BaseMorphism(RF.parent(), DO_P, mor)(RF)

In [49]:
to_integrate = -par_equ.coefficient_full(z_p[0]).coefficients()[0] / par_equ.coefficient_full(z_p[1]).coefficients()[0]
to_integrate

(x^10*t^3 - 5*x^9*t^2 + 10*x^8*t + 60*x^5*t^3 - 10*x^7 - 150*x^4*t^2 - 450*x^3*t + 900*t^3 + 450*x^2)/(x^10*t^2 - 5*x^9*t + 5*x^8 + 60*x^5*t^2 - 150*x^4*t + 150*x^3 + 900*t^2)

In [50]:
# when maple is properly installed this integrates the previous function
try:
    from sage.interfaces.maple import maple as Maple
    if not Maple.is_running():
        Maple._start()
    _HAS_MAPLE = True
except RuntimeError:
    _HAS_MAPLE = False

if _HAS_MAPLE:
    command = f"integrate({to_integrate}, x);"
    output = Maple.eval(command)
else:
    output = "No Maple found"

In [51]:
output

'No Maple found'

IOStream.flush timed out
IOStream.flush timed out


We obtain after integrating using Maple the following value for the integral:
$$xt - \ln(x^5 + 30) + \ln(t^2x^5 - 5tx^4 + 5x^3 + 30t^2),$$
hence we conclude that the right factor has as solution the following function:
$$e^{xt} \frac{t^2x^5 - 5tx^4 + 5x^3 + 30t^2}{x^5 + 30}.$$

Finally, to find the other 3 solutions (that will be the conjugate cases for $t$) we observe that $\lambda = t^4$, so if we change $t \mapsto \zeta_4 t^4$ for a fourth root of unity, we get the corresponding solutions:

$$\Phi_i(x) = e^{x\zeta_4^i t} \frac{\zeta_4^{2i}t^2 x^5 - 5 \zeta_4^itx^4 + 5x^3 + 30\zeta_4^2t^2}{x^5 + 30},$$

where $\Phi_4(x)$ was the solution obtained before.